# Analyse de toutes les expériences exportées

Ce notebook recharge chaque expérience contenant `backtest_metrics.csv` dans `exports`, trace ses comparaisons par période et imprime uniquement les metrics clés.

In [ ]:
from pathlib import Path
import importlib
import sys
import pandas as pd

PLUGIN_DIR = Path(r"C:\dev\factor_backtest")
if str(PLUGIN_DIR) not in sys.path:
    sys.path.insert(0, str(PLUGIN_DIR))

import func
importlib.reload(func)

from BacktestEngine import PtfBuilder, build_periods_from_breakpoints
from func import plot_performance_comparison

print("Plugin chargé")

In [ ]:
EXPORT_ROOT = PLUGIN_DIR / "exports"
COMPARISON_MAX_TESTS = 8  # Augmentez cette valeur seulement si chaque figure reste lisible.
PERIOD_BREAKPOINTS = [2009, 2013, 2017, 2020, 2022, 2024, 2026]
SHOW_PLOTS = True


In [ ]:
key_metric_columns = [
    "display_label", "test_path", "test_name", "test_type", "metric",
    "benchmark", "raw_variables", "composition_recipe",
    "period_label", "actual_start_date", "actual_end_date",
    "observation_count", "years",
    "robust_score", "robust_score_rank_global", "robust_score_rank_within_type",
    "top_total_return", "top_annualized_return",
    "top_annualized_volatility", "top_sharpe_ratio",
    "top_max_drawdown", "top_sortino_ratio", "top_beta",
    "top_tracking_error", "top_information_ratio",
    "worst_annualized_return", "bench_annualized_return",
    "top_bench_ratio", "top_worst_ratio", "active_max_drawdown",
    "tracking_error_annualized", "min_rolling_3y_cagr",
    "active_cagr", "top_worst_cagr",
]

def _relative_cagr(portfolio_return, reference_return):
    if pd.isna(portfolio_return) or pd.isna(reference_return):
        return float("nan")
    if 1 + reference_return <= 0:
        return float("nan")
    return (1 + portfolio_return) / (1 + reference_return) - 1


def _reconstruct_metrics_from_performances(experiment_dir, period_breakpoints):
    saved_metrics = pd.read_csv(experiment_dir / "backtest_metrics.csv")
    if "scope" in saved_metrics.columns:
        metadata_rows = saved_metrics.loc[
            saved_metrics["scope"].astype(str).eq("total")
        ]
    else:
        metadata_rows = saved_metrics
    metadata_by_path = (
        metadata_rows.drop_duplicates("test_path").set_index("test_path").to_dict(
            orient="index"
        )
    )
    sources = func._load_saved_performances(experiment_dir)
    metric_calculator = PtfBuilder.__new__(PtfBuilder)
    total_rows = []
    subperiod_rows = []
    required_columns = ["Top", "Worst", "Bench"]

    for test_path, source in sources.items():
        performance = source["performance"].copy()
        missing_columns = [
            column for column in required_columns if column not in performance.columns
        ]
        if missing_columns:
            raise KeyError(
                f"Performance absente pour {test_path} : {missing_columns}"
            )
        performance = performance.loc[:, required_columns].sort_index()
        performance.index = pd.to_datetime(performance.index, errors="coerce")
        performance = performance.loc[performance.index.notna()].dropna()
        if performance.empty:
            continue

        base = dict(metadata_by_path.get(test_path, {}))
        metadata = source.get("metadata", {})
        fallback_metadata = {
            "test_path": test_path,
            "test_name": source.get("test_name") or test_path,
            "test_type": metadata.get("test_type"),
            "metric": metadata.get("metric"),
        }
        for column, value in fallback_metadata.items():
            if column not in base or pd.isna(base[column]):
                base[column] = value
        base["test_path"] = test_path

        classic_metrics = {
            portfolio: metric_calculator._calculate_classic_metrics(
                performance[portfolio],
                benchmark=performance["Bench"],
            )
            for portfolio in required_columns
        }
        robust_score, top_bench_ratio, top_worst_ratio = (
            metric_calculator._calculate_robust_score(
                performance["Top"],
                performance["Worst"],
                performance["Bench"],
                store_metrics=True,
            )
        )
        robust_metrics = dict(metric_calculator.robust_metrics)
        total_row = {
            **base,
            "scope": "total",
            "period_id": "total",
            "period_label": "Période totale",
            "actual_start_date": performance.index[0].date().isoformat(),
            "actual_end_date": performance.index[-1].date().isoformat(),
            "observation_count": len(performance),
            "years": max((len(performance) - 1) / 252, 0),
            "top_cagr": classic_metrics["Top"]["annualized_return"],
            "worst_cagr": classic_metrics["Worst"]["annualized_return"],
            "bench_cagr": classic_metrics["Bench"]["annualized_return"],
            "active_cagr": _relative_cagr(
                classic_metrics["Top"]["annualized_return"],
                classic_metrics["Bench"]["annualized_return"],
            ),
            "top_worst_cagr": _relative_cagr(
                classic_metrics["Top"]["annualized_return"],
                classic_metrics["Worst"]["annualized_return"],
            ),
            "robust_score": robust_score,
            "top_bench_ratio": top_bench_ratio,
            "top_worst_ratio": top_worst_ratio,
            **robust_metrics,
        }
        for portfolio, metrics in classic_metrics.items():
            total_row.update({
                f"{portfolio.lower()}_{metric}": value
                for metric, value in metrics.items()
            })
        total_rows.append(total_row)

        period_metrics = metric_calculator._calculate_period_metrics(
            performance, period_breakpoints
        )
        for period_row in period_metrics.to_dict(orient="records"):
            subperiod_rows.append({
                **base,
                **period_row,
                "scope": "subperiod",
                "test_path": test_path,
            })

    reconstructed_metrics = pd.DataFrame([*total_rows, *subperiod_rows])
    return func._finalize_backtest_metrics(reconstructed_metrics)


def _period_definitions(period_breakpoints):
    return [
        {"id": "total", "label": "Période totale", "start": None, "end": None},
        *build_periods_from_breakpoints(period_breakpoints),
    ]


def _prepare_reconstructed_comparisons(experiment_dir, metrics, periods):
    comparisons = {}
    for period in periods:
        selections, ratio_definitions = func.build_performance_comparison_definitions(
            export_dir=experiment_dir,
            max_tests=COMPARISON_MAX_TESTS,
            period_id=period["id"],
            metrics=metrics,
        )
        performance, composition = func.combine_backtest_performances(
            export_dir=experiment_dir,
            selections=selections,
            return_composition=True,
        )
        ratios = func.calculate_performance_ratios(
            performance,
            benchmark_column="Benchmark",
            ratio_definitions=ratio_definitions,
        )
        comparisons[period["id"]] = {
            "performance": performance,
            "ratios": ratios,
            "composition": composition,
            "performance_selection": selections,
            "ratio_definitions": ratio_definitions,
            "period": period,
            "period_definitions": periods,
        }
    return comparisons

experiment_dirs = sorted(
    experiment_dir
    for experiment_dir in EXPORT_ROOT.iterdir()
    if experiment_dir.is_dir()
    and (experiment_dir / "backtest_metrics.csv").exists()
)
if not experiment_dirs:
    raise FileNotFoundError(
        f"Aucun dossier d'expérience avec backtest_metrics.csv dans {EXPORT_ROOT}"
    )

period_definitions = _period_definitions(PERIOD_BREAKPOINTS)
comparisons_by_period_by_experiment = {}
comparison_figures = {}

print("\n" + "=" * 100)
print(f"EXPORT_ROOT : {EXPORT_ROOT}")
print(f"PERIOD_BREAKPOINTS : {PERIOD_BREAKPOINTS}")
print("Périodes reconstruites :")
print(", ".join(period["label"] for period in period_definitions))
print(f"Nombre d'expériences : {len(experiment_dirs)}")
print("Metrics clés recalculées à partir des performance CSV")
print("=" * 100)

for experiment_dir in experiment_dirs:
    experiment_name = experiment_dir.name
    print("\n" + "@" * 100)
    print(f"EXPÉRIENCE : {experiment_name}")
    print(f"EXPORT_DIR : {experiment_dir}")

    prompt_metrics = _reconstruct_metrics_from_performances(
        experiment_dir, PERIOD_BREAKPOINTS
    )
    comparisons_by_period = _prepare_reconstructed_comparisons(
        experiment_dir, prompt_metrics, period_definitions
    )
    comparisons_by_period_by_experiment[experiment_name] = comparisons_by_period

    experiment_figures = {}
    for period_id, comparison in comparisons_by_period.items():
        experiment_figures[period_id] = plot_performance_comparison(
            performance=comparison["performance"],
            ratios=comparison["ratios"],
            benchmark_column="Benchmark",
            title=f"{experiment_name} | Comparaison des performances",
            save_path=None,
            show_plot=SHOW_PLOTS,
            rebase=True,
            show_worst_performance=False,
            period_definitions=comparison["period_definitions"],
            default_period_id=period_id,
        )
    comparison_figures[experiment_name] = experiment_figures

    for period_id, comparison in comparisons_by_period.items():
        period = comparison["period"]
        selected_top = [
            (label, test_path)
            for label, (test_path, portfolio)
            in comparison["performance_selection"].items()
            if portfolio == "Top"
        ]
        print("\n" + "#" * 100)
        print(f"PERIOD_ID : {period_id}")
        print(f"Période : {period['label']}")
        print(f"Début réel : {period.get('start')}")
        print(f"Fin réelle : {period.get('end')}")
        if not selected_top:
            print("Aucune performance Top sélectionnée.")
            continue

        selected_paths = [test_path for _, test_path in selected_top]
        display_labels = {label_path: label for label, label_path in selected_top}
        period_metrics = prompt_metrics.loc[
            prompt_metrics["period_id"].astype(str).eq(str(period_id))
            & prompt_metrics["test_path"].isin(selected_paths)
        ].copy()
        if not period_metrics.empty:
            period_metrics["_selection_order"] = period_metrics["test_path"].map(
                {test_path: index for index, test_path in enumerate(selected_paths)}
            )
            period_metrics["display_label"] = period_metrics["test_path"].map(
                display_labels
            )
            period_metrics = period_metrics.sort_values("_selection_order").drop(
                columns="_selection_order",
            )
        available_metric_columns = [
            column for column in key_metric_columns if column in period_metrics.columns
        ]
        print("Facteurs Top sélectionnés et metrics clés (CSV) :")
        print(period_metrics[available_metric_columns].to_csv(index=False))